In [ ]:
pip install tensorflow

In [ ]:
pip install opencv-python

In [ ]:
!git clone https://github.com/krasserm/super-resolution   #Cloning a Pretrained Model

fatal: destination path 'super-resolution' already exists and is not an empty directory.


In [ ]:
cd /content/super-resolution #Creation of Directory

[Errno 2] No such file or directory: '/content/super-resolution #Creation of Directory'
/content


In [ ]:
import os
print(os.path.exists('/content/super-resolution'))


True


In [ ]:
!tar xvfz /content/weights-srgan.tar.gz

weights/srgan/gan_discriminator.h5
weights/srgan/gan_generator.h5
weights/srgan/pre_generator.h5


# **SRGAN Model Implementation :-**

---



---



In [ ]:
import os
print(os.listdir('.'))


['.config', 'super-resolution', 'sample_video.mp4', 'Cute_Panda_eating_grass.mp4', 'data', 'weights', 'weights-srgan.tar.gz', 'sample_data']


In [ ]:
from tensorflow.keras.layers import Add, BatchNormalization, Conv2D, Dense, Flatten, Input, LeakyReLU, PReLU, Lambda


In [ ]:
!sed -i 's/tensorflow\.python\.keras/tensorflow.keras/g' /content/super-resolution/model/srgan.py


In [ ]:
!wget -O sample_video.mp4 "https://www.learningcontainer.com/wp-content/uploads/2020/05/sample-mp4-file.mp4"


--2025-04-23 18:44:51--  https://www.learningcontainer.com/wp-content/uploads/2020/05/sample-mp4-file.mp4
Resolving www.learningcontainer.com (www.learningcontainer.com)... 104.21.16.1, 104.21.32.1, 104.21.80.1, ...
Connecting to www.learningcontainer.com (www.learningcontainer.com)|104.21.16.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10546620 (10M) [video/mp4]
Saving to: ‘sample_video.mp4’

sample_video.mp4    100%[===================>]  10.06M  --.-KB/s    in 0.1s    

2025-04-23 18:44:51 (87.1 MB/s) - ‘sample_video.mp4’ saved [10546620/10546620]



In [ ]:
# Importing all necessary libraries
import timeit
import cv2
import os
import numpy as np
import sys
sys.path.append('/content/super-resolution')
from model  import resolve_single
from utils import load_image, plot_sample
from model.srgan import generator

# Read the video from specified path
cam = cv2.VideoCapture("/content/Cute_Panda_eating_grass.mp4")
fps = cam.get(cv2.CAP_PROP_FPS)
print(fps)


try:

    # Creating a folder named data
    if not os.path.exists('data'):
        os.makedirs('data')

# if not created then raise error
except OSError:
    print ('Error: Creating directory of data')

#frames Extraction from video
currentframe = 0
arr_img = []
while(True):

    # reading from frame
    ret,frame = cam.read()

    if ret:
        # if video is still left continue creating images
        name = './data/frame' + str(currentframe).zfill(3) + '.jpg'
        print ('Creating...' + name)

        # writing the extracted images
        cv2.imwrite(name, frame)

        # increasing counter so that it will show how many frames are created
        currentframe += 1
        #storing the path of extracted frames in a list
        arr_img.append(name)
    else:
        break
#print(arr_img)

start = timeit.default_timer()
model = generator()
model.load_weights('weights/srgan/gan_generator.h5')

#Initialization of an empty list to store the super resolved images
arr_output=[]
print(len(arr_img))
n= len(arr_img)

#Implementation of SRGAN Model in extracted frames
for i in range(n):
  lr = load_image(arr_img[i])
  sr = resolve_single(model, lr)
  #plot_sample(lr, sr)

  arr_output.append(sr)
stop = timeit.default_timer()
#print(arr_output)

print("time : ", stop-start)

# Release all space and windows once done
cam.release()
cv2.destroyAllWindows()

8.0
Creating..../data/frame000.jpg
Creating..../data/frame001.jpg
Creating..../data/frame002.jpg
Creating..../data/frame003.jpg
Creating..../data/frame004.jpg
Creating..../data/frame005.jpg
Creating..../data/frame006.jpg
Creating..../data/frame007.jpg
Creating..../data/frame008.jpg
Creating..../data/frame009.jpg
Creating..../data/frame010.jpg
Creating..../data/frame011.jpg
Creating..../data/frame012.jpg
Creating..../data/frame013.jpg
Creating..../data/frame014.jpg
Creating..../data/frame015.jpg
16
time :  131.61102510299997


# **Saving the Super Resolved Frames :-**
The super resolved frames are in numpy array format, so here we will change them in an image format.

In [ ]:
# Importing necessary libraries
from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array
from keras.preprocessing.image import array_to_img
from keras.preprocessing.image import save_img
import os

# Define output directory path
output_dir = "/content/super-resolution/output_images"
os.makedirs(output_dir, exist_ok=True)  # ✅ Create the directory if it doesn't exist

# Initialization of an empty list to store the path of Super resolved frames
s_res = []
for j in range(len(arr_output)):
    out_name = os.path.join(output_dir, 'frame' + str(j).zfill(3) + '.jpg')
    img_pil = array_to_img(arr_output[j])
    save_img(out_name, img_pil)
    s_res.append(out_name)

# print(s_res)


In [ ]:
import os
from keras.preprocessing.image import array_to_img, save_img

# Define output directory path
output_dir = "/content/super-resolution/output_images"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# Initialize an empty list to store the paths of super-resolved frames
s_res = []

for j in range(len(arr_output)):
    out_name = f"{output_dir}/frame{str(j).zfill(3)}.jpg"
    img_pil = array_to_img(arr_output[j])
    save_img(out_name, img_pil)  # Save the image
    s_res.append(out_name)

print("Super-resolved frames saved successfully in:", output_dir)


Super-resolved frames saved successfully in: /content/super-resolution/output_images


# **Conversion of Super Resolved frames into a video**

In [ ]:
import cv2
import numpy as np
for i in range(len(s_res)):
    filename=s_res[i]
    #reading each files
    img = cv2.imread(filename)
    height, width, layers = img.shape
    size = (width,height)

fps =10
 #Put the fps value as your convenience or
               #Calculate by using (No. of frames)/Video_duration in seconds

#Creation of output video
out = cv2.VideoWriter('drama2_output.mp4',cv2.VideoWriter_fourcc(*'DIVX'), fps , size)

#Writing Frames into video
for i in range(len(s_res)):
    out.write(cv2.imread(s_res[i]))
out.release()

In [ ]:
import cv2
import os

# Define the output video directory and create it if it doesn't exist
video_output_dir = "/content/super-resolution/output_videos"
os.makedirs(video_output_dir, exist_ok=True)

# Define output video path
output_video_path = os.path.join(video_output_dir, "panda2.mp4")

# Initialize video size using the first valid image
size = None
for filename in s_res:
    img = cv2.imread(filename)
    if img is not None:
        height, width, layers = img.shape
        size = (width, height)
        break

if size is None:
    raise ValueError("None of the images could be read. Check the paths in s_res.")

fps = 10

out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'DIVX'), fps, size)

# Writing frames to the video
for filename in s_res:
    img = cv2.imread(filename)
    if img is not None:
        out.write(img)
    else:
        print(f"Warning: Failed to read {filename}")

out.release()
print(f"🎬 Video successfully created at: {output_video_path}")


🎬 Video successfully created at: /content/super-resolution/output_videos/panda2.mp4
